# CSC AI Summer School 2026 - day 4: MLOps tutorial

### Task: train an object detection model to detect the second metacarpal in hand X-rays

In this tutorial, we will train an AI model to automatically locate the second metacarpal (the long bone in the hand connecting the wrist to the index finger, which we will call M2) in hand X-ray images. 

The model that we train is an **object detection** model. It will learn to draw an oriented bounding box (a rotated rectangle) tightly around the bone, when it believes it's found the M2 bone.

Here's an outline of what we will do:

1. Setting up our working environment (turning on the GPU, installing libraries and modules, downloading the dataset)
2. Exploring the data
3. Setting up our experiment logging system, mlflow
4. Training the object detection model
5. Examining the learning curves (loss and metrics as a function of training epochs) for the trained model
6. Evaluate the performance on the independent test set

This tutorial is mostly running as-is the cells (they are pre-filled). However some cells will have missing content that you should be able to fill. When a cell requires content to be filled, there will be <font color="red">a note in red</font> above the cell, and you will need to complete the cell. You will also be provided with the completed tutorial in case you prefer to just run the cells and follow what's happening.

---
## 1 — Setup

We'll follow a few steps to enable a GPU for accelerated training, download the dataset, and install all required libraries and Python modules.

### 1.1 — Enable the GPU runtime

Google Colab gives you a free GPU that makes training ~50× faster than CPU — we just need to enable it first!

1. In the top menu: press **Runtime** and select **Change runtime type**
2. Under **Hardware accelerator**, select **T4 GPU**
3. Click **Save**.
4. Click **Connect** on the top right side of the page, just under the share button.

A GPU check will run automatically after the libraries are installed in step 1.4.


### 1.2 — Download the dataset

Run the cell below to download the dataset directly to your Colab VM.

By default, we'll download the data to a folder named `content`.


In [ ]:
%pip install gdown --quiet
import gdown, zipfile
from pathlib import Path

Path('./content').mkdir(exist_ok=True) # ensure the content folder exists for the dataset to be downloaded into
DATASET_ZIP = Path("./content/dataset.zip") # path of the zip file to be downloaded; this will be deleted after extraction.
DATASET_DIR = Path("./content/SS-day4-hand-xray-m2-dataset") # path of the extracted dataset. 

if not DATASET_DIR.exists():
    gdown.download(id="1eVS15hC39zlMMcc7SvTShQerqQLGHUu2", output=str(DATASET_ZIP), quiet=False)
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall("./content/")
    DATASET_ZIP.unlink()  # delete the zip to free up space
    print(f"Dataset ready at {DATASET_DIR}")
else:
    print(f"Dataset already exists at {DATASET_DIR} - skipping download.")


### 1.3 — Install libraries and import modules

Run the cell below to install (on the colab virtual machine) python modules that we will use to train a model. This takes about a minute on first run.

In [ ]:
%pip install ultralytics==8.4.52 mlflow==3.11.1 pyyaml --quiet


Then, run the cell below to import the remaining relevant modules into the current notebook:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Patch
import mlflow
import numpy as np
import os
from PIL import Image
import random
import torch
from ultralytics import YOLO
import yaml

Now, we want to make sure that the GPU is accessible and that we can train a model using it. The pytorch library has a simple method for checking if GPU(s) are available; if the cell below raises an error, it means that the GPU was not properly enabled at step 1.1.

In [ ]:
# Detect available hardware and set DEVICE for use throughout the notebook.
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"✓ CUDA GPU ready: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available(): # this script can also work on a macbook with an apple silicon chip!
    DEVICE = "mps"
    print("✓ Apple MPS GPU ready")
else:
    import platform
    if platform.system() == "Windows": # unfortunately, the script does not support windows!
        raise RuntimeError("Windows detected — please use Google Colab instead 🙏")
    raise RuntimeError(
        "No GPU detected. On Colab: Runtime → Change runtime type → T4 GPU → Save, then re-run.")


### 1.4 — Set configuration parameters

We are now setting up some parameters for model training.

We will train a state-of-the-art object detection model to detect the second metacarpal **only**. The general name of the model is YOLO (not "you only live once", but "you only look once"). 

More specifically, we will train the `yolov8n-obb` variant of YOLO, which is a lightweight YOLO model that supports oriented bounding boxes.

The model will be trained over 10 epochs, with an image size of 320 and a batch size of 8. We also fix the random seed so the experiment is fully reproducible.

<font color="red">Please fill some of the variables below according to the text above.</font>


In [ ]:
# First, give your experiment a meaningful name - this will be used to track the experiment with MLflow.
EXPERIMENT_NAME = "hand-xray-m2-obb"

# Set up the parameters of the model, as described above
MODEL_NAME  = "yolov8n-obb.yaml"
EPOCHS      = ???   # number of training epochs
IMG_SIZE    = ???   # image size in pixels
BATCH_SIZE  = ???   # images per batch
SEED        = ???   # a random integer

# Set random seeds for reproducibility.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# The YOLO model is trained via the Ultralytics package, which provides a high-level API for training and evaluating YOLO models.
# The package expects the data to be organized in a certain way, and it also needs to know where the data is and how it's structured.

# This is done by passing a .yaml file that contains this information to the model. We create this .yaml file here:

# Create the .yaml file content as a dictionary:
yaml_content = {
    "path" : str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val"  : "val/images",
    "test" : "test/images",
    "nc"   : ???,        # number of classes
    "names": {0: ???},   # name of the class (a string, e.g. "MyClass")
}

YAML_PATH = DATASET_DIR / "data.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_content, f)
    print(f"Created the .yaml file at {YAML_PATH}.")

---
## 2 — Exploring the Data

### Dataset

We use a 🔗 **[publicly available dataset](https://universe.roboflow.com/hf-w1rwi/hand-xray)** from Roboflow containing 1,153 hand X-ray images annotated with various bones (M2, M3, M4, radius, ulna) as oriented bounding boxes. We will focus only on the second metacarpal (M2).

Before training any model, it is crucial to look at the data. Below we will:

- Display a sample of X-ray images
- Overlay the ground-truth bounding boxes so we can see what the model is being asked to learn

### Label format

To look into the data, we need to understand how the labels (oriented bounding boxes for the bones) are presented. 

Within the dataset directory (`DATASET_DIR`), we have the folders `train`, `val` and `test` which respectively contain the preallocated data splits for training, validation and independent testing. Within each of those three folders, the actual X-rays can be found within the `images` folder as `.jpg` files. Each image has a matching `.txt` file within the `labels` folder. Every line in that file describes one bounding box:

```
class  x1 y1  x2 y2  x3 y3  x4 y4
```

The eight numbers are the **four corners** of the oriented rectangle, expressed as fractions of the image width and height (values between 0 and 1).

To read the images and superimpose the labels on top of them, we define two helper functions:

In [ ]:
def load_obb_labels(label_path: Path) -> list:
    """Return a list of [x1,y1,x2,y2,x3,y3,x4,y4] rows (normalised 0-1)."""
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 9:
                boxes.append([float(v) for v in parts[1:]])
    return boxes


def draw_obb(ax, coords: list, image_w: int, image_h: int, colour="red", linewidth=2):
    """Draw a single oriented bounding box on a matplotlib axes.

    coords  : 8 normalised floats [x1,y1,x2,y2,x3,y3,x4,y4]
    image_w : pixel width  of the image
    image_h : pixel height of the image
    """
    pts = np.array(coords).reshape(4, 2)
    pts[:, 0] *= image_w
    pts[:, 1] *= image_h
    polygon = patches.Polygon(
        pts, closed=True,
        edgecolor=colour, facecolor="none", linewidth=linewidth
    )
    ax.add_patch(polygon)

Using the two helper functions, we can display some example data from the training set. You can re-run this cell to look at new examples every time.

<font color="red">Here, you have to fill the image loading procedure with the PIL library.</font>

In [ ]:
train_img_dir = DATASET_DIR / "train" / "images"
train_lbl_dir = DATASET_DIR / "train" / "labels"

all_img_paths = sorted(train_img_dir.glob("*.jpg"))
sample_paths  = random.sample(all_img_paths, 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_paths):
    img  = ... # Here, you need to find how to open the image with the PIL library (Image.open....)
    w, h = img.size
    boxes = load_obb_labels(train_lbl_dir / (img_path.stem + ".txt"))

    ax.imshow(img)
    for box in boxes:
        draw_obb(ax, box, w, h, colour="lime", linewidth=2)
    ax.set_title(img_path.stem[:30], fontsize=8)
    ax.axis("off")

fig.suptitle("Sample training images — green box = second metacarpal (M2)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3 — Setting up MLflow

**MLflow** is an open-source tool for tracking machine learning experiments. It records metrics (like loss and accuracy) at every training epoch so you can compare runs and spot problems like overfitting.

We store the MLflow data in a folder called `mlruns/` on this Colab machine. Ultralytics (the python module that manages the YOLO model that we train) will detect it automatically and log metrics there during training. After training we will display the results directly in this notebook.

In [ ]:
# First, we need to tell MLflow where we want to store the experiment data:

MLFLOW_DIR = "mlruns" # this is just a local folder ./mlruns within colab; this could also point to a database or a remote server.
Path(MLFLOW_DIR).mkdir(exist_ok=True) # Create the above folder
mlflow.set_tracking_uri(MLFLOW_DIR) # This is telling the MLflow client where to read/write the data. 
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_DIR # By also setting this environment variable, Ultralytics is made aware of where the data gets logged.

# Similarly, we also need to tell MLflow as well as Ultralytics what is the name of our experiment for proper tracking. This was defined in 1.5:
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_experiment(EXPERIMENT_NAME)
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT_NAME  # groups runs under one experiment name



---
## 4 — Training the YOLO Model

We use the **Ultralytics YOLO** library to train the model. 

We've already defined, in section 1.4, some parameters that we want to use for the model (number of epochs, model architecture, image size and batch size). We pass this information to the model as we're calling the `train` method.

When you run this cell, you should see some text appearing showing the training at each epoch - training should be completed within a few minutes!

<font color="red">Here, you have to fill model training parameters as we've defined them in section 1.</font>

In [ ]:
# Instantiate YOLO model - by passing yolov8n-obb.yaml we're asking the model to load the yolov8n-obb architecture and train from scratch (random weights).
model = YOLO(MODEL_NAME)

# Train model, passing training parameters. 
results = model.train(
    data     = str(YAML_PATH),
    epochs   = ???,
    fraction = 0.3,        # leave as is - train on 30% of the data for speed
    imgsz    = ???,
    batch    = ???,
    cache    = False,      # leave as is
    project  = "runs",     # leave as is
    name     = "m2-obb",   # leave as is
    exist_ok = True,       # leave as is
    seed     = SEED,       # leave as is
    device   = DEVICE,     # leave as is
    verbose  = False,
)

RESULTS_DIR = Path(results.save_dir)
print("\nTraining complete!")
print(f"Results saved to: {RESULTS_DIR}")


---
## 5 — Training Results

Once the model is trained (or as we're training it) we want to see if it trained well. For that, we can pull the per-epoch metrics that were logged into MLflow.

We're going to evaluate three metrics, defined here:

(1) **box loss**: this is the mathematical function that the model is optimising for. The model tries to minimise the box loss at every epoch - the box loss is higher when the original bounding box of the M2 differs from the predicted one. Over epochs, this value should always be decreasing for the training set.

(2) **precision**: this checks the accuracy of our model's positive predictions. Out of all the objects that the model detected, what fraction were really the M2?

(3) **recall**: this checks the completeness of the model. Out of all M2 in the dataset, what fraction did we detect?

To look at this data, we will connect to our MLflow experiment, read the relevant metrics from it, and plot the results.

<font color="red">Here, you'll have select what variables to plot to see how metrics evolve as a function of epochs.</font>

In [ ]:
client = mlflow.MlflowClient()

# MLflow will log experiments under EXPERIMENT_NAME; each time you run model.train(), you will create what is called a new "run" within that experiment,
# such that one experiment can contain multiple runs. Here, we load the experiment called EXPERIMENT_NAME, sort the runs by start time, and 
# get the most recent run:
runs   = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], order_by=["start_time DESC"])
run_id = runs.iloc[0]["run_id"]

# We write a quick helper function to parse the results stored into mlflow in something readable. 
def get_metric_series(run_id, metric_name):
    history = client.get_metric_history(run_id, metric_name)
    return [m.step for m in history], [m.value for m in history]

# Ultralytics logs into mlflow with specific names - here we're reading the training and validation loss, as well as the precision and recall
# for the validation set, as calculated on the bounding boxes (hence the B):
loss_epochs,  train_box_loss = get_metric_series(run_id, "train/box_loss")
_,            val_box_loss   = get_metric_series(run_id, "val/box_loss")
prec_epochs,  precision      = get_metric_series(run_id, "metrics/precisionB")
rec_epochs,   recall         = get_metric_series(run_id, "metrics/recallB")

# Finally, we plot the results:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(???, ???, marker="o", label="training set", color="steelblue")
axes[0].plot(???, ???,   marker="o", label="validation set",   color="orange")
axes[0].set_title("Box Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(???, ???, marker="o", color="green")
axes[1].set_title("Precision (on the validation set)"); axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)

axes[2].plot(???, ???, marker="o", color="purple")
axes[2].set_title("Recall (on the validation set)"); axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()


A side note related to the above: 

The YOLO model outputs, with every detected object, a confidence score. The YOLO model will find a lot of objects in each image, but will discard a few of them that have a poor confidence score.

By default, the metrics presented (precision and recall) are calculated with the default confidence score of 0.25. By raising the confidence threshold (say to 0.5), less objects may be detected, although the ones that are detected are more likely to be correct. In our example, both precision and recall are excellent with the default confidence score. In a practical implementation, the threshold may require tuning to achieve your goals. 

---
## 6 — Inference and Evaluation on the Test Set

We've now looked at the results in step 5 on the validation set - hopefully you are satisfied with the performance of the model!

As a final step, we take the best model that we've trained and we evaluate it on the test set, which are images that the model has never seen during training or validation. By "best model" we mean the one across epochs with the lowest validation loss.

Ideally, we want to have similar performance on the test set compared to the validation set - if performance significantly drops, there may be concerns of overfitting. 

To evaluate on the final set, we are going to load the best trained model (the one with the lowest validation loss), and apply it on the test set. We're also going to re-apply it on the train and validation set and compare the numbers we get.

<font color="red">Here, we have to figure out what method of `best_model()` to call to evaluate the performance on a labeled split. As you've seen in section 4, to train a model, we called `model.train()` - now we need to call something else to perform evaluation. Here are some methods of model: `predict`, `train`, `val`, `export`, `track`. Can you pick which one you should use here?</font>

In [ ]:
# Load the best model
best_model = YOLO(str(RESULTS_DIR / "weights" / "best.pt"))

# Evaluate the model separately on training, validation and testing sets. This may take a minute.
# The val(). method evaluates results and returns metrics for when you have labels. 
train_val = best_model.???(data=str(YAML_PATH), split="train", verbose=False)
val_val   = best_model.???(data=str(YAML_PATH), split="val", verbose=False)
test_val  = best_model.???(data=str(YAML_PATH), split="test", verbose=False)

print(f"-" * 60)
print(f"The precision on training, validation and test sets are: {train_val.box.mp:.2f}, {val_val.box.mp:.2f}, {test_val.box.mp:.2f}.")
print(f"The recall on training, validation and test sets are: {train_val.box.mr:.2f}, {val_val.box.mr:.2f}, {test_val.box.mr:.2f}.")

Based on the recall/precision values above, you can probably get a sense of the model's capacity to find the M2 bone.

However, this does not tell us the whole story - we don't know how well the estimated boxes overlap with the labels.

While there are some metrics to evaluate that, we will instead do a visual inspection of some examples to get an idea if the model is performing as intended.

<font color="red">Here, fill the image loading procedure similarly to what you've done in section 2.</font>

<font color="red">Similarly to the above, we also need to decide what method of best_model() should be used to perform inference on new images. Can you decide the correct option out of `predict`, `train`, `val`, `export`, and `track`? </font>


In [ ]:
# Get the paths for the test set data
test_img_dir = DATASET_DIR / "test" / "images"
test_lbl_dir = DATASET_DIR / "test" / "labels"

# Randomly sample 6 images for visualisation
test_imgs    = random.sample(sorted(test_img_dir.glob("*.jpg")), 6)

# Plot the images, labels and predicted boxes.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, img_path in zip(axes, test_imgs):
    img  = ??? # open image with PIL
    w, h = img.size

    gt_boxes = load_obb_labels(test_lbl_dir / (img_path.stem + ".txt"))
    pred = best_model.???(str(img_path), verbose=False)[0]

    ax.imshow(img)
    for box in gt_boxes:
        draw_obb(ax, box, w, h, colour="lime", linewidth=2)
    if pred.obb is not None and len(pred.obb.xyxyxyxyn) > 0: # if there are predictions,
        for pred_box in pred.obb.xyxyxyxyn.cpu().numpy(): # bring back each prediction to cpu,
            draw_obb(ax, pred_box.flatten(), w, h, colour="red", linewidth=2) # and draw!
    ax.set_title(img_path.stem[:30], fontsize=7)
    ax.axis("off")


fig.legend(
    handles=[Patch(edgecolor="lime", facecolor="none", label="Ground truth"),
             Patch(edgecolor="red",  facecolor="none", label="Prediction")],
    loc="lower center", ncol=2, fontsize=11, frameon=False
)
fig.suptitle("Test set predictions", fontsize=13)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

---
## Conclusion

Well done, you've trained a YOLO oriented bounding box model to automatically detect the second metacarpal in hand X-rays!

This concludes the tutorial for now. Feel free to play around with the script and change some parameters, especially in the model training, and see how it impacts performance.